# 03. 로컬 검색 인덱스와 벡터 공간 관리

목표: Embedding API로 만들 수 있는 대표 기능인 로컬 문서 검색을 축소 구현한다.

실행 방법:
1. 위에서 아래로 셀을 실행한다.
2. 문서 임베딩과 질의 임베딩의 `taskType`을 분리하는 이유를 확인한다.
3. 벡터 공간 식별자가 다르면 검색을 막는 방어 코드를 살펴본다.

이 노트북은 Python 표준 라이브러리만 사용한다. 실제 브라우저 저장소는 IndexedDB나 OPFS를 사용해야 한다.

In [ ]:
import hashlib
import math
import re

TOKEN_RE = re.compile(r"[a-zA-Z0-9가-힣]+")


def tokenize(text):
    return [token.lower() for token in TOKEN_RE.findall(text)]


def cosine_similarity(vec_a, vec_b):
    if len(vec_a) != len(vec_b):
        raise ValueError("Vectors must have the same dimension")
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

## 1. 의미를 조금 더 반영하는 장난감 임베딩

이전 노트북의 해시 벡터는 의미 검색에는 부적합하다. 여기서는 작은 키워드 그룹을 만들어 로컬 검색 흐름을 이해할 수 있게 한다.

In [ ]:
CONCEPTS = {
    "browser_ai": {"chrome", "browser", "built", "built-in", "ai", "model", "models", "canary"},
    "privacy": {"privacy", "private", "local", "on-device", "device", "offline"},
    "search": {"search", "retrieval", "query", "document", "index", "vector"},
    "performance": {"latency", "fast", "performance", "download", "available", "cost"},
    "feedback": {"feedback", "issue", "bug", "github", "chromium"},
}


def concept_embedding(text, task_type):
    """작은 개념 사전을 이용해 텍스트를 벡터로 바꾼다.

    실제 Embedding API는 모델이 의미를 학습하지만, 여기서는 규칙으로 흉내 낸다.
    taskType을 약하게 반영해 document와 query가 다른 경로로 생성되었음을 보존한다.
    """
    tokens = set(tokenize(text))
    values = []
    for words in CONCEPTS.values():
        overlap = len(tokens & words)
        values.append(float(overlap))

    # taskType별로 아주 작은 바이어스를 둔다. 제품에서는 이 정보를 메타데이터로 저장해야 한다.
    if task_type == "retrieval-query":
        values = [value * 1.05 for value in values]
    elif task_type == "retrieval-document":
        values = [value * 0.95 for value in values]

    norm = math.sqrt(sum(value * value for value in values)) or 1.0
    return [value / norm for value in values]

## 2. 로컬 문서 인덱스 만들기

Embedding API explainer는 API가 벡터 DB를 제공하지 않는다고 설명한다. 따라서 개발자가 직접 IndexedDB, OPFS, 서버 DB 같은 저장소에 벡터와 메타데이터를 저장해야 한다.

In [ ]:
documents = [
    {"id": "doc-1", "title": "Canary setup", "text": "Enable the semantic embedder flag in Chrome Canary."},
    {"id": "doc-2", "title": "Privacy", "text": "On-device embeddings keep raw text local to the user's device."},
    {"id": "doc-3", "title": "Search", "text": "Use retrieval-document for indexed passages and retrieval-query for user searches."},
    {"id": "doc-4", "title": "Feedback", "text": "File API feedback in the GitHub explainer or technical bugs in Chromium."},
    {"id": "doc-5", "title": "Model readiness", "text": "Wait until availability returns available before calling create."},
]

VECTOR_SPACE_ID = "mock-embedding-space-v1"


def build_index(documents, vector_space_id):
    index = []
    for doc in documents:
        vector = concept_embedding(doc["title"] + " " + doc["text"], "retrieval-document")
        index.append({
            "id": doc["id"],
            "title": doc["title"],
            "text": doc["text"],
            "values": vector,
            "taskType": "retrieval-document",
            "spaceId": vector_space_id,
        })
    return index


index = build_index(documents, VECTOR_SPACE_ID)
index[0]

## 3. 질의 검색하기

검색 질의는 `retrieval-query` 용도로 임베딩하고, 문서 인덱스는 `retrieval-document` 용도로 만든다. 둘은 같은 검색 시스템 안에서 짝을 이루는 태스크다.

In [ ]:
def search(index, query, vector_space_id, top_k=3):
    query_vector = concept_embedding(query, "retrieval-query")
    results = []

    for item in index:
        # 같은 공간에서 나온 벡터만 비교한다. 이 검사는 실제 제품에서 중요하다.
        if item["spaceId"] != vector_space_id:
            raise RuntimeError("Vector space mismatch. Rebuild the index before searching.")
        score = cosine_similarity(query_vector, item["values"])
        results.append({"score": score, "title": item["title"], "text": item["text"]})

    return sorted(results, key=lambda row: row["score"], reverse=True)[:top_k]


for row in search(index, "How do I search local documents with embeddings?", VECTOR_SPACE_ID):
    print(f"{row['score']:.3f} | {row['title']} | {row['text']}")

## 4. 모델 또는 벡터 공간 변경 시 인덱스 무효화

원문은 같은 공간에서 나온 벡터만 비교하라고 경고한다. 브라우저 모델이 업데이트되거나 API가 다른 차원을 반환하면 기존 인덱스를 다시 만들어야 한다.

In [ ]:
try:
    search(index, "privacy local device", "mock-embedding-space-v2")
except RuntimeError as error:
    print("검색 차단:", error)

print("운영 정책: spaceId가 바뀌면 저장된 문서 벡터를 재생성한다.")

## 5. 브라우저 제품으로 옮길 때의 저장 형태

브라우저에서는 아래와 비슷한 레코드를 IndexedDB나 OPFS에 저장한다. 원문 청크와 벡터뿐 아니라 `taskType`, `spaceId`, 생성 시간도 함께 저장해야 디버깅과 재색인이 가능하다.

In [ ]:
record = {
    "chunkId": "note-17#chunk-03",
    "sourceId": "note-17",
    "text": "On-device embeddings keep raw text local.",
    "values": concept_embedding("On-device embeddings keep raw text local.", "retrieval-document"),
    "taskType": "retrieval-document",
    "spaceId": VECTOR_SPACE_ID,
    "createdAt": "2026-07-17T00:00:00+09:00",
}

record

## 정리

- Embedding API는 벡터 생성 API이지 벡터 DB가 아니다.
- 문서 인덱스는 `retrieval-document`, 사용자 질의는 `retrieval-query`로 분리한다.
- 벡터 저장 시 `spaceId`, `taskType`, 차원, 생성 시점을 함께 기록한다.
- 공간 식별자가 바뀌면 검색을 막고 인덱스를 재생성하는 정책이 필요하다.